# Занятие 5, пара 2. Предобработка. Дубликаты, выбросы, проверки

Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai

## Таблица выглядит аккуратно. Значит, готова?

Числовые поля уже разобрали. Но одинаковая заявка всё ещё может учитываться дважды, «высшее» и «Высшее» — становиться разными категориями, а редкий дорогой курс — исчезнуть просто за то, что он редкий.

Сегодня разберём три решения: какие повторы можно удалить, какие значения надо **исследовать**, а не автоматически исправлять, и как закрепить договорённости проверками. Результат — воспроизводимая таблица, отдельные спорные записи и объяснение, что было потеряно или сохранено.

Нужны операции 05.1: маски, `.loc`, строковый тип, преобразование чисел, смысл пропуска. Переменные из другой тетради не нужны: начинаем в новом ядре.

### Подготовка

Основные примеры работают без «Ваш ход». Пропускаете задание — пропустите и его `check(...)`, затем продолжайте обычный раздел. Решения упражнений не нужны для демонстрации. Сохраняйте свои ответы в отдельной копии.

Нужны Pandas и Matplotlib, сеть не требуется. Служебная ячейка ниже подключает `course_support` у текущей папки или её родителей. `DATA_DIR` — учебные исходники, `WORKSPACE_DIR` — ваша рабочая папка. Эту настройку не нужно писать по памяти.

In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Продолжим с контрольной копии

В `requests_stage1.csv` лежит результат основного пути 05.1 без упражнений. Файл входит в комплект: соседняя преподавательская папка не нужна. У ID и исходных полей снова явно укажем строковый тип. Числовые `age`, `hours`, `fee_rub` уже сохранены в пригодном для чтения виде; пустые числовые поля при чтении останутся пропусками.

Свои ответы из 05.1 не заменяем. Если хотите продолжить именно собственный результат, после прохождения можно отдельно поменять путь на `reports/preprocessing/requests_stage1.csv` и сравнить.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

stage_path = DATA_DIR / 'preprocessing' / 'requests_stage1.csv'
text_types = {'request_id': 'string', 'age_raw': 'string',
              'duration_raw': 'string', 'fee_raw': 'string',
              'education': 'string', 'unit': 'string', 'format': 'string'}
stage = pd.read_csv(stage_path, dtype=text_types)
display(stage.head())
print('Записей:', len(stage))

### 2. Совпало всё — посмотрим на повтор

`duplicated()` сравнивает значения всех столбцов. По умолчанию первую строку оставляет, последующие совпадения отмечает True. `keep=False` отмечает все участники повторения — так удобно посмотреть на пару до удаления. Служебный индекс строк при этом не сравнивается.

В этом источнике повтор одной и той же записи заявки не является новым событием. Для журнала действий два одинаковых события могли бы быть нормальны, поэтому метод сам не решает смысловую задачу.

In [ ]:
exact_mask = stage.duplicated(keep=False)
display(stage.loc[exact_mask])
no_exact = stage.drop_duplicates()
no_exact = no_exact.copy()
print('После точных повторов:', len(no_exact))

### 3. Для человека одинаково, для строки — нет

Посмотрим `value_counts(dropna=False)`: метод считает частоты разных значений, а `dropna=False` просит не скрывать пропуски. «Высшее» и « высшее » пока разные категории. Компьютер не обязан догадаться, что мы имели в виду.

In [ ]:
display(no_exact['education'].value_counts(dropna=False))

Начнём с одной строки. `.strip()` убирает пробелы по краям, `.lower()` переводит буквы в нижний регистр. Сначала выполним отдельно: так видно, что именно изменилось.

In [ ]:
label = ' Высшее '
without_edges = label.strip()
lower_label = without_edges.lower()
print(repr(label))
print(repr(without_edges))
print(repr(lower_label))

На Series такие операции доступны через `.str`. А сокращение «высш.» регистром не исправить: нужен словарь замен. `.replace({'высш.': 'высшее'})` меняет только указанное значение целиком. Мы берём это соответствие из SOURCE.md. «Похожее слово» не всегда значит ту же категорию.

In [ ]:
education = no_exact['education'].str.strip()
education = education.str.lower()
education = education.replace({'высш.': 'высшее'})
normalized = no_exact.copy()
normalized['education'] = education
display(normalized['education'].value_counts(dropna=False))

### ✏️ Ваш ход — нормализовать и не объединить лишнее

Создайте `my_labels` из `[' Высшее', 'ВЫСШ.', 'СПО ', 'среднее', None]` с `dtype='string'`. Получите `my_normalized`: убрать внешние пробелы, понизить регистр и заменить «высш.» на «высшее». СПО и среднее оставьте разными, пропуск не превращайте в слово «нет».

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('05.2.1', my_normalized)

**Мой вывод:** что изменили, на каком основании и что осталось неизвестным. …

### 4. Теперь некоторые строки действительно совпали

После унификации категории повтор 002 стал точным. Удалим полные повторы ещё раз, но сначала посмотрим их. Исходный `stage` остаётся нетронутым, поэтому всегда можно восстановить написание до нормализации.

In [ ]:
normalized_repeats = normalized.duplicated(keep=False)
display(normalized.loc[normalized_repeats])
unique_records = normalized.drop_duplicates().copy()
print('После нормализации и точных повторов:', len(unique_records))

Остались ли повторяющиеся ID? Теперь используем `duplicated(subset=['request_id'], keep=False)`: сравнивается только ключ. Две разные записи с одним ключом — конфликт, если по договорённости на заявку положена одна версия.

`drop_duplicates('request_id', keep='last')` не означает «взять правильное». Последняя строка в файле не обязательно самая актуальная, а времени обновления в нашем источнике нет.

In [ ]:
conflict_mask = unique_records.duplicated(subset=['request_id'], keep=False)
conflicts = unique_records.loc[conflict_mask].copy()
display(conflicts[['request_id', 'fee_raw', 'fee_rub']])

Для ID 016 две цены, и источник не позволяет выбрать. Сохраним обе строки в `conflicts`, а в рабочую `clean` пока возьмём непротиворечивые заявки. Знак `~` инвертирует булеву маску: True становится False и наоборот. Это временное исключение из анализа с объяснением, не удаление следов.

Если вопрос касается всех заявок, в выводе нужно указать неполноту. Когда источник уточнит правильную цену, запись можно вернуть.

In [ ]:
clean = unique_records.loc[~conflict_mask].copy()
print('Основная таблица:', len(clean))
print('На уточнение:', len(conflicts))
print('Ключ основной таблицы уникален:', clean['request_id'].is_unique)

### 5. Большое число — ошибка или интересное наблюдение?

В стоимости есть 300000. Можно сразу выкинуть, чтобы среднее стало симпатичнее. Но вопрос «почему здесь большое число?» полезнее вопроса «каким методом его удалить?»

Сначала познакомимся с квартилями на игрушечном ряду. `.quantile(0.25)` — первый квартиль Q1, `.quantile(0.75)` — третий Q3; Pandas по умолчанию линейно интерполирует между соседними позициями, если нужное место не целое. **IQR = Q3 − Q1** — межквартильный размах. Правило 1.5 × IQR отмечает значения дальше Q1−1.5IQR или Q3+1.5IQR. Это эвристический сигнал для исследования, не доказательство ошибки.

In [ ]:
demo_prices = pd.Series([1000, 2000, 3000, 4000, 20000])
q1_demo = demo_prices.quantile(0.25)
q3_demo = demo_prices.quantile(0.75)
iqr_demo = q3_demo - q1_demo
lower_demo = q1_demo - 1.5 * iqr_demo
upper_demo = q3_demo + 1.5 * iqr_demo
print(q1_demo, q3_demo, iqr_demo)
print('Границы сигнала:', lower_demo, upper_demo)

Получили −1000 и 7000. Отрицательная нижняя граница не означает, что отрицательная цена стала допустимой по правилам источника. Статистический сигнал и проверка предметного смысла — разные проверки.

### ✏️ Ваш ход — посчитать сигнал, а не приговор

Для `my_prices = pd.Series([10, 12, 14, 16, 100])` вычислите `my_lower`, `my_upper` по правилу 1.5IQR и `my_suspects` — значения вне границ. Не удаляйте их из исходной Series. Напишите, какую дополнительную информацию захотели бы получить про 100.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('05.2.2', my_lower, my_upper, my_suspects)

**Мой вывод:** что изменили, на каком основании и что осталось неизвестным. …

Применим тот же расчёт к рабочим заявкам. `quantile` пропускает неизвестные значения. Это не делает покрытие полным: неизвестные цены остаются неизвестными.

In [ ]:
q1 = clean['fee_rub'].quantile(0.25)
q3 = clean['fee_rub'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outlier_mask = (clean['fee_rub'] < lower) | (clean['fee_rub'] > upper)
display(clean.loc[outlier_mask, ['request_id', 'fee_rub', 'format']])

Источник подтвердил индивидуальный пакет 012. Его цена остаётся в `clean`, но для вопроса о **групповых** занятиях выберем нужный формат. Мы меняем состав анализа по смыслу вопроса, а не «лечим» число, которое портит среднее.

Среднее и медиана отвечают на разные вопросы. Ни одна не является автоматически правильнее; сообщим обе и число известных цен. При таком маленьком вымышленном наборе нельзя делать выводы о настоящем рынке.

In [ ]:
group_mask = clean['format'] == 'групповой'
group_courses = clean.loc[group_mask]
print('Все форматы: среднее', clean['fee_rub'].mean())
print('Все форматы: медиана', clean['fee_rub'].median())
print('Групповой: среднее', group_courses['fee_rub'].mean())
print('Групповой: известных цен', group_courses['fee_rub'].notna().sum())

**Опыт на 6–8 минут.** На отдельной копии замените только цену 012 на 30000. Предскажите, куда изменятся среднее и медиана, затем проверьте. Результат эксперимента не записывайте обратно в `clean`: цель — увидеть чувствительность показателя, а не придумать более удобную стоимость.

### 6. Как не забыть, о чём мы договорились

Код выполняется без ошибок — ещё не значит, что данные правильные. Зафиксируем несколько конкретных свойств результата. `.is_unique` проверяет уникальность столбца; `.notna().all()` — что все значения известны; `.all()` у булевой Series требует True для каждого элемента.

Для неотрицательной цены сначала выберем известные значения через `.dropna()`: неизвестная цена разрешена. Отдельно проверим, что таблица не пустая — иначе многие проверки «все элементы подходят» вернут True даже без единой строки.

In [ ]:
known_fees = clean['fee_rub'].dropna()
nonnegative = known_fees >= 0
quality = {
    'есть строки': len(clean) > 0,
    'ID известны': bool(clean['request_id'].notna().all()),
    'ID уникальны': clean['request_id'].is_unique,
    'известные цены неотрицательны': bool(nonnegative.all())
}
print(quality)

`bool(...)` делает обычное логическое значение Python. Теперь остановим выполнение, если свойство нарушено. `assert условие, 'сообщение'` пропускает нас дальше при True и вызывает AssertionError при False. Это удобная учебная самопроверка, но не полноценная система контроля входных данных сервиса.

Проверки не обязаны быть ровно четыре или пять. Их число определяется смыслом источника. Здесь мы проверяем то, о чём договорились, а не заполняем отчёт галочками.

In [ ]:
assert quality['есть строки'], 'Таблица неожиданно пустая'
assert quality['ID известны'], 'Есть запись без ID'
assert quality['ID уникальны'], 'Остался конфликт ключа'
assert quality['известные цены неотрицательны'], 'Есть отрицательная цена'
print('Договорённости основной таблицы выполнены')

Покажем ожидаемую ошибку на отдельной Series. `try/except` уже встречался в парсинге: здесь ловим именно намеренную демонстрацию AssertionError, чтобы продолжить тетрадь. Так не надо скрывать неожиданную ошибку всего конвейера.

In [ ]:
broken_fees = pd.Series([1000, -50])
try:
    assert (broken_fees >= 0).all(), 'Отрицательная цена требует разбора'
except AssertionError as error:
    print('Ожидаемая остановка:', error)

### ✏️ Ваш ход — проверка должна замечать плохие данные

Напишите `my_quality(table)`, которая возвращает True, только если таблица непустая, `request_id` не содержит пропусков и уникален, все известные `fee_rub` неотрицательны. Пропуски цены разрешены. Соедините условия оператором `and`. Проверка попробует как хорошие, так и плохие примеры — постоянное `return True` не поможет.

После запуска добавьте в свою пробную таблицу повтор ID или отрицательную цену. Объясните, какое именно правило сработало.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('05.2.3', my_quality)

**Мой вывод:** что изменили, на каком основании и что осталось неизвестным. …

### 7. Сохраним не только хороший результат

Сохраним основную таблицу и конфликтные записи отдельно. Исходник и промежуточный файл уже есть, поэтому выбор можно пересмотреть. При чтении результата вновь явно зададим строковый ID — CSV не хранит схему типов DataFrame.

Метод `.tolist()` превращает столбец в обычный список значений. Сравнение таких списков проверяет и сами ID, и их порядок. Оно не заменяет последующую проверку числовых полей.


In [ ]:
output_dir = WORKSPACE_DIR / 'reports' / 'preprocessing'
output_dir.mkdir(parents=True, exist_ok=True)
clean_path = output_dir / 'requests_clean.csv'
conflict_path = output_dir / 'requests_conflicts.csv'
clean.to_csv(clean_path, index=False, encoding='utf-8')
conflicts.to_csv(conflict_path, index=False, encoding='utf-8')
restored = pd.read_csv(clean_path, dtype={'request_id': 'string'})
assert len(restored) == len(clean), 'Изменилось число записей'
assert restored['request_id'].tolist() == clean['request_id'].tolist(), 'Изменились ключи'
print(clean_path)
print(conflict_path)

Одного совпадения числа строк недостаточно: сохраняемый формат мог потерять значение или единицу. Выберем столбцы, важные для анализа, и проверим числовые данные. `pd.testing.assert_frame_equal` сравнивает две таблицы и сообщает, где отличие. `check_dtype=False` здесь разрешает различие технического числового типа после CSV, но не подмену значений; исходный строковый ID проверен отдельно.

In [ ]:
important = ['age', 'hours', 'fee_rub']
pd.testing.assert_frame_equal(restored[important], clean[important], check_dtype=False)
print('Числа и пропуски после чтения согласованы')

### Ваш собственный набор: небольшой следующий шаг

**Самостоятельно, 8–10 минут.** Для выбранных данных проекта составьте короткую таблицу решений: поле → возможная проблема → проверка → действие → что сохраняем. Если данных ещё нет, используйте наш набор, но выберите другой вопрос: например, какие длительности групповых курсов встречаются в выгрузке.

Не нужно специально находить пять дефектов. Достаточно одной настоящей проблемы или обоснованной проверки, которая показала отсутствие проблемы. Не подменяйте «ошибок не нашли» утверждением «ошибок быть не может».

До следующего блока сохраните тетрадь с объяснениями и отдельную копию. Эти решения пригодятся при очистке данных проекта; новые баллы за сегодняшние упражнения не вводятся. GitHub необязателен.

### Что осталось у нас

Повтор записи не равен двум похожим людям. Конфликт ключа не лечится выбором «последнего» без правил источника. Выброс — повод выяснить причину. Проверка закрепляет конкретную договорённость. Теперь можно переходить к группировкам и объединению таблиц: чистые ключи и одинаковые категории там особенно пригодятся.

Для чтения по желанию: [глава 7 книги Уэса Маккинни — подготовка данных](https://wesmckinney.com/book/data-cleaning.html), разделы о дубликатах и преобразовании значений. Художественная отсылка — уже знакомый «Человек, который изменил всё»: обсудить, как выбор показателя меняет решение; фильм не заменяет проверку данных.